# Household MPC Experiment

Dedicated paired baseline-versus-income-shock workflow. The implementation lives in `src/experiments/mpc.py`; the historical command remains available through `run_mpc_experiment.py`.

In [ ]:
# Setup
%load_ext autoreload
%autoreload 2

import pandas as pd

from src.experiments.mpc import run_mpc_experiment
from src.mpc_analysis import MPCFilterConfig, plot_mpc_panel


## Inputs

In [ ]:
RUN_MPC = False
COUNTRY = "FRA"
SEEDS = [121, 224, 432, 214, 445, 256, 157, 858, 2, 43, 54, 333, 323, 223, 3554, 4546]
T_MAX = 30
SHOCK_PERIOD = 20
HORIZON_PERIODS = 4
SHOCK_FRACTION = 0.1
N_JOBS = 4
OUTPUT_DIR = "data/output_data/mpc-experiment"

filters = MPCFilterConfig(
    min_income=1000,
    income_top_quantile=0.99,
    consumption_income_ratio_quantiles=(0.01, 0.99),
    gross_wealth_top_quantile=0.999,
    head_age_min=25,
    head_age_max=75,
)


## Run

In [ ]:
outputs = None
if RUN_MPC:
    outputs = run_mpc_experiment(
        seeds=SEEDS,
        t_max=T_MAX,
        shock_period=SHOCK_PERIOD,
        horizon_periods=HORIZON_PERIODS,
        shock_fraction=SHOCK_FRACTION,
        country_iso3=COUNTRY,
        output_dir=OUTPUT_DIR,
        n_jobs=N_JOBS,
        force_rebuild_data=True,
        apply_mpc_filters=True,
        mpc_filter_config=filters,
        stay_in_activity_bracket_only=False,
        stable_effective_labor_state_only=True,
    )
else:
    print("Set RUN_MPC = True to run the experiment.")


## Results and plots

In [ ]:
if outputs is not None:
    panel = pd.read_csv(outputs["analysis_dir"] / "household_mpc_panel.csv")
    summary = pd.read_csv(outputs["analysis_dir"] / "household_mpc_summary.csv")
    plot_mpc_panel(
        panel,
        variables=[
            "income", "net_wealth", "housing_tenure",
            "net_liquid_financial_assets", "illiquid_financial_assets", "housing_wealth",
        ],
        mpc_columns=["mpc_impact", "cmpc_4q"],
        plot_kind="box_only",
        y_range=(-0.5, 1.1),
        winsor_quantiles=(0.01, 0.99),
        winsor_mode="trim",
        output_dir=outputs["analysis_dir"] / "plots",
        output_format="png",
    )
    summary
